In [2]:
!pip install -q zarr numcodecs xarray dask


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 110.5 MB/s eta 0:00:00


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import xarray as xr
xr.backends.list_engines()


{'h5netcdf': <H5netcdfBackendEntrypoint>
   Open netCDF (.nc, .nc4 and .cdf) and most HDF5 files using h5netcdf in Xarray
   Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.H5netcdfBackendEntrypoint.html,
 'scipy': <ScipyBackendEntrypoint>
   Open netCDF files (.nc, .cdf and .nc.gz) using scipy in Xarray
   Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.ScipyBackendEntrypoint.html,
 'store': <StoreBackendEntrypoint>
   Open AbstractDataStore instances in Xarray
   Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.StoreBackendEntrypoint.html,
 'zarr': <ZarrBackendEntrypoint>
   Open zarr files (.zarr) using zarr in Xarray
   Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.ZarrBackendEntrypoint.html}

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import xarray as xr
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

DATA_PATH = "/content/drive/MyDrive/Indian Weather Prediction/datasets/processed.zarr"
MODEL_PATH = "/content/drive/MyDrive/Indian Weather Prediction/models/convlstm_final.pt"

VARS = ["t2m", "d2m", "sp", "msl", "u10", "v10", "tp"]
INPUT_STEPS = 4
BATCH_SIZE = 8

# --------------------
# MODEL
# --------------------
class ConvNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(7, 32, 3, padding=1)
        self.relu = nn.ReLU()
        self.temp_head = nn.Conv2d(32, 1, 1)
        self.rain_head = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        x = x[:, -1]
        x = self.relu(self.conv(x))
        return self.temp_head(x), self.rain_head(x)

model = ConvNet().to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

ds = xr.open_zarr(DATA_PATH)

TEST_YEARS = ["2015", "2016", "2017", "2018", "2022", "2023"]
results = []

# --------------------
# SAFE BACKTESTING
# --------------------
for year in TEST_YEARS:
    print(f"Backtesting year {year}...")

    ds_y = ds.sel(valid_time=slice(f"{year}-01-01", f"{year}-12-31"))

    mse_t = mae_t = corr_num_t = corr_den_t = 0
    mse_r = mae_r = corr_num_r = corr_den_r = 0
    mse_t_base = mse_r_base = 0
    n = 0

    for i in range(INPUT_STEPS, len(ds_y.valid_time)):
        x = np.stack(
            [ds_y[v].isel(valid_time=slice(i-INPUT_STEPS, i)).values for v in VARS],
            axis=1
        )
        x = torch.tensor(x[None], dtype=torch.float32).to(device)

        with torch.no_grad():
            pt, pr = model(x)

        yt = ds_y["t2m"].isel(valid_time=i).values
        yr = ds_y["tp"].isel(valid_time=i).values

        pt = pt.squeeze().cpu().numpy()
        pr = pr.squeeze().cpu().numpy()

        bt = ds_y["t2m"].isel(valid_time=i-1).values
        br = ds_y["tp"].isel(valid_time=i-1).values

        mse_t += np.mean((yt - pt) ** 2)
        mae_t += np.mean(np.abs(yt - pt))
        mse_r += np.mean((yr - pr) ** 2)
        mae_r += np.mean(np.abs(yr - pr))

        mse_t_base += np.mean((yt - bt) ** 2)
        mse_r_base += np.mean((yr - br) ** 2)

        n += 1

    rmse_t = np.sqrt(mse_t / n)
    rmse_r = np.sqrt(mse_r / n)
    mae_t /= n
    mae_r /= n

    skill_t = 1 - (rmse_t / np.sqrt(mse_t_base / n))
    skill_r = 1 - (rmse_r / np.sqrt(mse_r_base / n))

    results.append([year, rmse_t, mae_t, skill_t, rmse_r, mae_r, skill_r])

# --------------------
# RESULTS
# --------------------
df = pd.DataFrame(
    results,
    columns=[
        "Year",
        "Temp_RMSE", "Temp_MAE", "Temp_Skill",
        "Rain_RMSE", "Rain_MAE", "Rain_Skill"
    ]
)

print("\nBACKTESTING RESULTS:\n")
print(df)
print("\nAVERAGE PERFORMANCE:\n")
print(df.mean(numeric_only=True))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda
Backtesting year 2015...
Backtesting year 2016...
Backtesting year 2017...
Backtesting year 2018...
Backtesting year 2022...
Backtesting year 2023...

BACKTESTING RESULTS:

   Year  Temp_RMSE  Temp_MAE  Temp_Skill  Rain_RMSE  Rain_MAE  Rain_Skill
0  2015   0.272309  0.193348    0.123977   0.001087  0.000688   -0.805762
1  2016   0.273354  0.193908    0.132507   0.001092  0.000690   -0.849945
2  2017   0.273064  0.195283    0.128156   0.001118  0.000700   -0.798607
3  2018   0.280676  0.200106    0.118239   0.001119  0.000706   -0.886374
4  2022   0.281809  0.201396    0.112653   0.001112  0.000698   -0.806575
5  2023   0.274479  0.195477    0.131376   0.001072  0.000672   -0.789879

AVERAGE PERFORMANCE:

Temp_RMSE     0.275948
Temp_MAE      0.196586
Temp_Skill    0.124485
Rain_RMSE     0.001100
Rain_MAE      0.000692
Rain_Skill   -0.822857
